# SME-DataTrust BI — configuración
Ejecuta los notebooks del 01 al 06 en orden y dentro de la misma sesión de Colab. Si tu carpeta del proyecto está en otra ruta, modifica `PROJECT_DIR`.

In [ ]:
from pathlib import Path
import sys
candidates=[Path("/content/BI-PYMES-ML"),Path("/content/SME-DataTrust-BI"),Path.cwd()]
PROJECT_DIR=next((p for p in candidates if (p/"src").exists()),Path.cwd())
if not (PROJECT_DIR/"src").exists():
    raise FileNotFoundError("No se encontró src/. Ajusta PROJECT_DIR a la raíz del repositorio.")
sys.path.insert(0,str(PROJECT_DIR))
for d in ["data/raw","data/processed","data/experimental","results/figures","results/metrics"]:
    (PROJECT_DIR/d).mkdir(parents=True,exist_ok=True)
print("Proyecto:",PROJECT_DIR)

## 6. Evaluación experimental
Se inyectan anomalías sintéticas en una copia de las características por factura. Las etiquetas indican qué registros fueron alterados. Esto permite evaluar detección de alteraciones controladas, pero no equivale a validar todos los errores reales.

In [ ]:
import pandas as pd
from src.experimental import run_comparison
p=PROJECT_DIR/"data/processed/invoice_features.pkl"
if not p.exists(): raise FileNotFoundError("Ejecuta primero Notebook 03.")
invoice=pd.read_pickle(p)
SAMPLE_N=30000
sample=invoice.sample(min(SAMPLE_N,len(invoice)),random_state=42).reset_index(drop=True)
synthetic,if_res,lof_res,base_res,metrics=run_comparison(sample,contamination=.03,random_state=42)
display(metrics)
metrics.to_csv(PROJECT_DIR/"results/metrics/synthetic_detection_metrics.csv",index=False)
synthetic.to_csv(PROJECT_DIR/"data/experimental/invoices_with_synthetic_anomalies.csv",index=False)

In [ ]:
from src.kpi_calculation import kpi_sensitivity
from src.anomaly_detection import fit_isolation_forest
_,original_scored,_=fit_isolation_forest(sample,contamination=.03,random_state=42)
lines=pd.read_pickle(PROJECT_DIR/"data/processed/retail_lines.pkl")
selected=set(sample.InvoiceNo.astype(str))
lines_sample=lines[lines.InvoiceNo.astype(str).isin(selected)].copy()
sensitivity=kpi_sensitivity(lines_sample,original_scored)
display(sensitivity)
sensitivity.to_csv(PROJECT_DIR/"results/metrics/kpi_sensitivity.csv",index=False)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4)); plt.bar(metrics.model,metrics.f1)
plt.ylabel("F1-score"); plt.title("Comparación en alteraciones sintéticas etiquetadas")
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(PROJECT_DIR/"results/figures/model_f1_comparison.png",dpi=160); plt.show()

### Información que debe reportarse
Registra el tamaño de la muestra, el método de muestreo, el porcentaje y mecanismo de inyección, parámetros, Precision, Recall y F1. Explica falsos positivos, falsos negativos, limitaciones de las anomalías sintéticas y por qué una anomalía no es automáticamente un error.